In [ ]:
%pip -q install gradio cmake ninja

In [ ]:
!nvidia-smi

In [ ]:
import os

os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["CUDACXX"] = "/usr/local/cuda/bin/nvcc"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
os.environ["FORCE_CMAKE"] = "1"

%pip install --no-cache-dir llama-cpp-python

In [ ]:
from google.colab import drive
import os
import sqlite3

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/LocalLLM"
MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "qwen3-27b",
    "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-IQ2_M.gguf",
)
CHAT_DB = os.path.join(BASE_DIR, "chat_history.sqlite3")

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found in Google Drive: {MODEL_PATH}")
if not os.path.isfile(CHAT_DB):
    raise FileNotFoundError(f"Chat database not found in Google Drive: {CHAT_DB}")

with sqlite3.connect(CHAT_DB) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

print("Model file:", MODEL_PATH)
print("Chat database:", CHAT_DB)

In [ ]:
import llama_cpp
from llama_cpp import Llama

if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError(
        "llama-cpp-python was installed without CUDA support. "
        "Re-run the CUDA installation cell in a GPU-enabled Colab runtime."
    )

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=18432,
    verbose=False,
)
print("Qwen model loaded with GPU offload enabled.")

In [ ]:

import sqlite3
import uuid
import gradio as gr

SYSTEM_PROMPT = (
    "You are a helpful AI assistant. "
    "Give clear, accurate and thoughtful answers."
)


# ---------- Persistent database ----------

def get_conversations():
    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT conversation_id, MAX(id) AS last_id
            FROM messages
            GROUP BY conversation_id
            ORDER BY last_id DESC
        """).fetchall()
    return [row[0] for row in rows]


def get_messages(cid):
    if not cid:
        return []

    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT role, content
            FROM messages
            WHERE conversation_id = ?
            ORDER BY id
        """, (cid,)).fetchall()

    # Always return Gradio's role/content message format.
    return [
        {"role": role, "content": content}
        for role, content in rows
        if role in ("user", "assistant")
    ]


def save_message(cid, role, content):
    with sqlite3.connect(CHAT_DB) as con:
        con.execute("""
            INSERT INTO messages (conversation_id, role, content)
            VALUES (?, ?, ?)
        """, (cid, role, content))


# ---------- UI actions ----------

def new_conversation():
    cid = str(uuid.uuid4())[:8]
    choices = get_conversations()

    if cid not in choices:
        choices.insert(0, cid)

    # Include the new ID in the choices before selecting it.
    return cid, [], gr.update(choices=choices, value=cid)


def load_conversation(cid):
    if not cid:
        return "", []

    return cid, get_messages(cid)


def chat(message, cid):
    message = (message or "").strip()
    cid = (cid or "").strip()

    if not message:
        return "", get_messages(cid), cid, gr.update(
            choices=get_conversations(), value=cid or None
        )

    if not cid:
        cid = str(uuid.uuid4())[:8]

    # Persist user input first.
    save_message(cid, "user", message)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + get_messages(cid)

    try:
        result = llm.create_chat_completion(
            messages=messages,
            temperature=0.7
        )

        answer = (
            result["choices"][0]["message"].get("content") or ""
        ).strip()

        if not answer:
            answer = "(The model returned an empty response.)"

        save_message(cid, "assistant", answer)

    except Exception as e:
        print("Model inference error:", repr(e))
        # Preserve the user's message and show the error in the UI.
        answer = f"Model error: {type(e).__name__}: {e}"

    choices = get_conversations()
    if cid not in choices:
        choices.insert(0, cid)

    return (
        "",
        get_messages(cid) + (
            [{"role": "assistant", "content": answer}]
            if answer.startswith("Model error:") else []
        ),
        cid,
        gr.update(choices=choices, value=cid)
    )


# ---------- Build interface ----------

with gr.Blocks(title="My AI Assistant") as demo:
    gr.Markdown("# My AI Assistant")
    gr.Markdown("Qwen3.8-27B · Chat history saved in Google Drive")

    with gr.Row():
        cid = gr.Textbox(
            label="Conversation ID",
            placeholder="Create a new chat or select a saved one",
            scale=3
        )
        new_btn = gr.Button("New chat")

    saved_chats = gr.Dropdown(
        label="Saved conversations",
        choices=get_conversations(),
        interactive=True
    )

    chatbot_ui = gr.Chatbot(
        height=500,
        label="Chat"
    )

    message = gr.Textbox(
        label="Your message",
        placeholder="Message your AI...",
        lines=2
    )

    send_btn = gr.Button("Send", variant="primary")

    new_btn.click(
        new_conversation,
        outputs=[cid, chatbot_ui, saved_chats]
    )

    saved_chats.change(
        load_conversation,
        inputs=saved_chats,
        outputs=[cid, chatbot_ui]
    )

    send_btn.click(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    message.submit(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    demo.load(
        lambda: ("", []),
        outputs=[cid, chatbot_ui]
    )

demo.launch(share=True, theme=gr.themes.Soft())
